# Setup and data
Mount Drive, unpack the code-only archive from the README, and inspect the transfer plan.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
from pathlib import Path
import subprocess, sys, os
DATA = Path(os.getenv("V3_DATA", "/content/drive/MyDrive/audio-v3"))
REPO = Path(os.getenv("V3_REPO", "/content/v3-src"))
LOCAL = Path(os.getenv("V3_LOCAL", "/content/v3-local"))
REPO.mkdir(parents=True, exist_ok=True)
subprocess.run(["tar","-xf",str(DATA/"v3-code.tar"),"-C",str(REPO)], check=True)
sys.path.insert(0, str(REPO))
from colab.run_stage import run
EXECUTE = False  # Set True after reviewing this stage's data and compute estimate.


In [ ]:
if EXECUTE:
    subprocess.run([sys.executable,"-m","pip","install","-r",str(REPO/"pipeline_v3/requirements-colab.txt")],check=True)


In [ ]:
run("setup", REPO, DATA, LOCAL)
subprocess.run([sys.executable,"-m","pipeline_v3.fetch","--data",str(DATA),"--dry-run"],cwd=REPO,check=True)

In [ ]:
if EXECUTE:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    subprocess.run([sys.executable,"-m","pipeline_v3.fetch","--data",str(DATA),"--sources",
                    "mailabs","mlaad","kathbath","indicsynth","indicvoices","xlsr","wavlm-base","--execute"],cwd=REPO,check=True)
run("prepare",REPO,DATA,LOCAL,EXECUTE)

Review development/selection.csv produced by the matcher. Check the language, speaker, source/reference voice and held-family counts before packing. Supplement missing groups with approved development originals and matched fakes.

In [ ]:
run("pack", REPO, DATA, LOCAL, EXECUTE)